In [1]:
import numpy as np
import estimators
import forward
import utils
import matplotlib.pyplot as plt

import estimators.moments
import estimators.laplace
import estimators.PBI
import forward.laplace
import utils.dimensional_reduction
import utils.errors
import utils.invlap
import utils.load_btcs
import utils.synthetic

In [2]:
seed = 1
Nt = 150
n_btcs, ts_m, btcs_m, xs_m, names_list, T_ = utils.load_btcs.data('data/river_solute_tracer_data.csv',complete_zeros = True, min_dimless_res = 1/Nt) # As downloaded from TIERRAS

In [3]:
memory_func = 'first order'
bound_cond = 'infinite'

# Methods not requiring forward evaluations

In [4]:
%%time
params_lap = np.array([estimators.laplace.ADEMF_1D(ts_m[i], btcs_m[i], xs_m[i], memory_func, bound_cond) for i in range(n_btcs)])
params_mom = np.array([estimators.moments.TSM(ts_m[i], btcs_m[i], xs_m[i], bound_cond) for i in range(n_btcs)])

CPU times: total: 7min 39s
Wall time: 2min 3s


In [5]:
errors_lap = np.array([utils.errors.compute_errors(ts_m[i], btcs_m[i], xs_m[i], params_lap[i], memory_func, bound_cond) for i in range(n_btcs)])
errors_mom = np.array([utils.errors.compute_errors(ts_m[i], btcs_m[i], xs_m[i], params_mom[i], memory_func, bound_cond) for i in range(n_btcs)])

# DSTE methods

## Prior parameter estimation

In [6]:
best_params_no_syn = params_lap.copy()
if memory_func == 'first order':
    best_params_no_syn[errors_mom[:,0]<errors_lap[:,0]] = params_mom[errors_mom[:,0]<errors_lap[:,0]]

In [7]:
mean_log_params, cov_log_params, cov_log_params_sqrt = utils.synthetic.generate_dist(best_params_no_syn[:,1:])

## Synthetic dataset

In [8]:
n_synth = 1000
t_ = np.linspace(0,T_,int(Nt*T_)+1)[1:]
params_synth, btcs_synth = utils.synthetic.generate(seed,n_synth,t_,memory_func, bound_cond, mean_log_params, cov_log_params*2.0)

In [9]:
n_lmbds = 35
btcs_mean, phis,lambdas, Zs = utils.dimensional_reduction.KL_decomposition(btcs_synth, n_lmbds, 1/Nt)

## NNI & PBI

In [11]:
n_vs = 121
v_epsilons = np.linspace(0.9,1.5,n_vs)
reg_Zs = 10**np.linspace(-7,-3,9)
KL_coeffs_out = [estimators.PBI.KL_coeffs(ts_m[i], btcs_m[i], xs_m[i],t_, btcs_mean, phis, lambdas, v_epsilons) for i in range(n_btcs)]
Zs_meas_v = np.array([out[0] for out in KL_coeffs_out])
v_ests = np.array([out[1] for out in KL_coeffs_out])

In [12]:
params_NNI = np.array([estimators.PBI.estimate_params_NNI(params_synth, Zs, v_ests[i], Zs_meas_v[i], reg_v = 0) for i in range(n_btcs)])

In [13]:
params_PBI = np.array([estimators.PBI.estimate_params_PBI(params_synth, Zs, v_ests[i], Zs_meas_v[i], 3, reg_v = 0) for i in range(n_btcs)])

In [14]:
errors_NNI = np.array([utils.errors.compute_errors(ts_m[i], btcs_m[i], xs_m[i], params_NNI[i], memory_func, bound_cond) for i in range(n_btcs)])
errors_PBI = np.array([utils.errors.compute_errors(ts_m[i], btcs_m[i], xs_m[i], params_PBI[i], memory_func, bound_cond) for i in range(n_btcs)])

## KL-DNN-Forward

In [ ]:
import torch
import sklearn

In [ ]:
torch.set_default_device("cuda")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
to_torch = lambda x: torch.tensor(np.float32(x)).to(device)
to_cpu = lambda x: x.cpu().detach().numpy()
sqrt_used_lambdas = np.sqrt(lambdas[:n_lmbds])
sqrt_used_lambdas_torch = to_torch(sqrt_used_lambdas)
phis_torch = torch.tensor(np.float32(phis.copy())).to(device)
btcs_mean_torch = torch.tensor(np.float32(btcs_mean.copy())).to(device)

params_normalization = lambda params: np.linalg.solve(cov_log_params_sqrt,(np.log(params)-mean_log_params).T).T
params_denormalization = lambda params: np.exp((cov_log_params_sqrt@(params.T)).T+mean_log_params)

params_normalization_torch = lambda params: torch.linalg.solve(to_torch(cov_log_params_sqrt),((torch.log(params)-to_torch(mean_log_params)).T)).T
params_denormalization_torch = lambda params: torch.exp((to_torch(cov_log_params_sqrt)@(params.T)).T+to_torch(mean_log_params))

In [ ]:
X_denorm = params_rand
print(cov_log_params_sqrt.shape)
X = params_normalization(X_denorm)
print(X_denorm, params_denormalization(X))
y = Zs[:,:n_lmbds]
print(y.shape)

In [ ]:
%%time
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.1, random_state=42)

# Define the DNN model
class DNN(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(DNN, self).__init__()
        self.fc1 = torch.nn.Linear(input_size, hidden_size)
        self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
        self.fc3 = torch.nn.Linear(hidden_size, hidden_size)
        self.fc4 = torch.nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# Initialize model, loss function, and optimizer
input_size = X_train.shape[1]
hidden_size = 100
output_size = y_train.shape[1]
model = DNN(input_size, hidden_size, output_size).to(device)
loss_function = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters())


# Convert data to PyTorch tensors and move to GPU
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)

# Create DataLoader for training
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
g = torch.Generator(device=device)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100, shuffle=True, generator = g)

# Train the model
num_epochs = 1000
print_interval = X_train.shape[0]/100
losses = []
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i + 1) % print_interval == 0:
            print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(train_loader)}], Loss: {running_loss / print_interval:.8f}')
            losses.append(running_loss / print_interval)
            running_loss = 0.0

# Evaluate the model on the test set
with torch.no_grad():
    model.eval()
    outputs = model(X_test_tensor)
    loss = loss_function(outputs, y_test_tensor)
    mae = torch.nn.L1Loss()(outputs, y_test_tensor)
    print(f"Mean Absolute Error on Test Set: {mae.item()}")

plt.semilogy(list(range(num_epochs)),losses)
# plt.ylim(0.000001,0.01)
plt.grid(which='major')
plt.grid(which='minor',linewidth = 0.1)

In [ ]:
predictions_train = to_cpu(model(X_train_tensor))
predictions_test = to_cpu(model(X_test_tensor))
errors_train = np.linalg.norm(predictions_train-y_train, axis = 1)/(len(phis)*Nt**2)**(1/2)
errors_test =  np.linalg.norm(predictions_test-y_test, axis = 1)/(len(phis)*Nt**2)**(1/2)

In [ ]:
rand_test = np.random.randint(len(errors_test))
plt.plot(btcs_mean+phis.T@y_test[rand_test])
plt.plot(btcs_mean+phis.T@predictions_test[rand_test])
RMSE_func(t_,btcs_mean+phis.T@predictions_test[rand_test],btcs_mean+phis.T@y_test[rand_test]),np.linalg.norm(predictions_test[rand_test]-y_test[rand_test])/(len(phis)*150**2)**(1/2)

In [ ]:
fig = plt.figure()
ax = plt.gca()
ax.semilogy(np.linspace(0,1,len(errors_train)),sorted(errors_train),label = 'Errors train')
ax.semilogy(np.linspace(0,1,len(errors_test)),sorted(errors_test),label = 'Errors test')
ax.grid(which='major')

ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(decimals = 2,xmax = 1))
ax.yaxis.set_minor_formatter(matplotlib.ticker.PercentFormatter(decimals = 2,xmax = 1))

ax.tick_params(axis='y', which='major', labelsize=10)
ax.tick_params(axis='y', which='minor', labelsize=6)
for label in ax.get_yticklabels():
    label.set_verticalalignment('bottom')

ax.legend()
ax.set_ylim(0.00001,0.01)
ax.set_xlim(0,1)
ax.grid(which='major')
ax.grid(which='minor',linewidth = 0.1)
ax.set_xlabel('Cumulative fraction of fitted breakthrough curves')
ax.set_ylabel('Root-mean-square error', labelpad= 0)
print(geom_mean(errors_test))

In [ ]:
def KL_DNN_MAP_optimizer(Z,N_iter,lr, N_halfs_lr, prior):#, variance_NN):
    etas_sqrt_lmbds_bt = to_torch(Z).to(device)
    def loss_func(params_normal):
        return torch.sum((model(params_normal)-etas_sqrt_lmbds_bt)**2)#1/variance_NN*torch.sum((model(params_normal)-etas_sqrt_lmbds_bt)**2)+torch.sum((params_normal)**2)
    init_guess = to_torch(params_normalization(prior))   
    init_guess.requires_grad_(True)
    
    Loss_list = []
    for j in range(N_halfs_lr):
        optimizer = torch.optim.Adam([init_guess], lr=lr/2**(j))
        for i in range(N_iter//N_halfs_lr):
            z = loss_func(init_guess)
            optimizer.zero_grad()
            z.backward()
            torch.nn.utils.clip_grad_norm_(init_guess, max_norm=1.0)
            optimizer.step()    
            Loss_list.append(z.item())
            if i>60 and np.mean(np.exp(np.mean(np.log(Loss_list[-N_iter//30:]))))/np.exp(np.mean(np.log(Loss_list[-2*N_iter//30:-N_iter//30])))>0.99:
                    break
    return params_denormalization(to_cpu(init_guess))#,np.array(Loss_list)

In [ ]:
%%time
params_KL_DNN_MAP = np.array([np.concatenate(([vs_PBI[i_btc]],KL_DNN_MAP_optimizer(Zs_PBI[i_btc],1000,0.08,3,best_params_no_syn[i_btc][1:]))) for i_btc in range(n_btcs)])
RMSEs_KL_DNN_MAP, KL_divs_KL_DNN_MAP = calculate_errors(params_KL_DNN_MAP)

In [ ]:
RMSEs_KL_DNN_MAP_mat = np.zeros((n_btcs,len(vars_KL_DNN)))
KL_divs_KL_DNN_MAP_mat = np.zeros((n_btcs,len(vars_KL_DNN)))
params_KL_DNN_MAP = np.zeros((n_btcs,4))
for i_btc in range(n_btcs):

    t,btc,x = ts_m_0[i_btc],btcs_m_0[i_btc],xs_m[i_btc]
    params_est_i = []
    for i in range(len(vars_KL_DNN)):
        params_est = KL_DNN_MAP_optimizer_outputs[i_btc][i][0]
        btc_est = forward.laplace.ADEMF_1D(t[1:],[x],vs_PBI[i_btc],params_est[0],params_est[1],[1,params_est[2]], memory_func, bound_cond)[:,-1]
        
        RMSEs_KL_DNN_MAP_mat[i_btc][i] = (RMSE_func(t[1:],btc[1:],btc_est))
        KL_divs_KL_DNN_MAP_mat[i_btc][i] = (KL_div_func(t[1:],btc[1:],btc_est))
        params_est_i.append(np.concatenate(([vs_PBI[i_btc]],params_est)))
    params_KL_DNN_MAP[i_btc] = (params_est_i[np.argmin(RMSEs_KL_DNN_MAP_mat[i_btc])])
RMSEs_KL_DNN_MAP = np.min(RMSEs_KL_DNN_MAP_mat,axis = 1)
KL_divs_KL_DNN_MAP = KL_divs_KL_DNN_MAP_mat[:,np.argmin(KL_divs_KL_DNN_MAP_mat,axis = 1)]

## KL-DNN-Inverse

In [ ]:
torch.set_default_device("cuda")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:

def inverse_model_augmentation(n_augmentation,n_lmbds_inv,aug_fac):

    projected_points_best = np.array([projected_points[i_btc][args_PBI[i_btc]] for i_btc in range(n_btcs)])
    
    mean_rejections = np.mean((projected_points_best-Zs_PBI)[:,:n_lmbds_inv], axis = 0)
    covariance_rejections = np.cov((projected_points_best-Zs_PBI)[:,:n_lmbds_inv].T)
    
    y_denorm = params_rand
    y = params_normalization(np.repeat(y_denorm,n_augmentation, axis = 0))
    
    X = np.repeat(Zs[:,:n_lmbds_inv],n_augmentation, axis = 0)
    if n_augmentation>1:
        X[~(np.arange(len(X))%n_augmentation==0)] += aug_fac*np.random.multivariate_normal(mean_rejections,covariance_rejections, size = n_synth*(n_augmentation-1)) 
    # print(y.shape)
    
    # Check if CUDA is available
    torch.set_default_device("cpu")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.1, random_state=42)
    
    # Define the DNN model
    class DNN(torch.nn.Module):
        def __init__(self, input_size, hidden_size, output_size):
            super(DNN, self).__init__()
            self.fc1 = torch.nn.Linear(input_size, hidden_size)
            self.fc2 = torch.nn.Linear(hidden_size, hidden_size)
            self.fc3 = torch.nn.Linear(hidden_size, hidden_size)
            self.fc4 = torch.nn.Linear(hidden_size, output_size)
    
        def forward(self, x):
            x = torch.relu(self.fc1(x))
            x = torch.relu(self.fc2(x))
            x = torch.relu(self.fc3(x))
            x = self.fc4(x)
            return x
    
    # Initialize model, loss function, and optimizer
    input_size = X_train.shape[1]
    hidden_size = 100
    output_size = y_train.shape[1]
    model_inv = DNN(input_size, hidden_size, output_size).to(device)
    loss_function = torch.nn.MSELoss()
    optimizer = torch.optim.Adam(model_inv.parameters())
    
    
    # Convert data to PyTorch tensors and move to GPU
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).to(device)
    
    # Create DataLoader for training
    train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100*n_augmentation, shuffle=True)
    
    # Train the model
    num_epochs = 1000
    print_interval = X_train.shape[0]/(100*n_augmentation)
    losses = []
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, (inputs, targets) in enumerate(train_loader):
            optimizer.zero_grad()
            outputs = model_inv(inputs)
            loss = loss_function(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            if (i + 1) % print_interval == 0:
                # print(f'Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(train_loader)}], Loss: {running_loss / print_interval:.8f}')
                losses.append(running_loss / print_interval)
                running_loss = 0.0
    
    # Evaluate the model on the test set
    with torch.no_grad():
        model_inv.eval()
        outputs = model_inv(X_test_tensor)
        loss = loss_function(outputs, y_test_tensor)
        mae = torch.nn.L1Loss()(outputs, y_test_tensor)
        # print(f"Mean Absolute Error on Test Set: {mae.item()}")
    
    plt.semilogy(list(range(num_epochs)),losses)
    # plt.ylim(0.000001,0.01)
    plt.grid(which='major')
    plt.grid(which='minor',linewidth = 0.1)
    return model_inv

In [ ]:
%%time
model_inv = inverse_model_augmentation(1, n_lmbds, 0.0)

In [ ]:
params_KL_DNN_inverse = np.concatenate((np.array(vs_PBI)[:,None],params_denormalization(to_cpu(model_inv(to_torch(Zs_PBI))))),axis = 1)
RMSEs_KL_DNN_inverse, KL_divs_KL_DNN_inverse = calculate_errors(params_KL_DNN_inverse)

In [ ]:
%%time
n_augmentation,n_lmbds_inv,aug_fac = 30, 10, 0.5
model_inv_aug = inverse_model_augmentation(n_augmentation,n_lmbds_inv,aug_fac)

In [ ]:
params_aug_KL_DNN_inverse = np.concatenate((np.array(vs_PBI)[:,None],params_denormalization(to_cpu(model_inv_aug(to_torch(np.array(Zs_PBI)[:,:n_lmbds_inv]))))),axis = 1)
RMSEs_aug_KL_DNN_inverse, KL_divs_aug_KL_DNN_inverse = calculate_errors(params_aug_KL_DNN_inverse)

# LIPO

In [ ]:
%%time
from lipo import GlobalOptimizer
vs_eps_min, vs_eps_max = 0.9,1.5
params_LIPO = np.zeros((n_btcs,4))
KL_divs_LIPO = np.zeros((n_btcs))
RMSEs_LIPO = np.zeros((n_btcs))
btcs_LIPO = []
lipo_errors_mat = []
params_LIPO_steps = []
num_function_calls = 1000
for i_btc in range(n_btcs):
    lipo_errors_list = []
    params_step = []
    t,btc,x = ts_m_0[i_btc],btcs_m_0[i_btc],xs_m[i_btc]
    def opt_function(v_eps,Pe,beta_prime, mem_param):
        params_step.append([v_eps,Pe,beta_prime, mem_param])
        if t[0]==0:
            est_bt =  np.concatenate(([0],forward.laplace.ADEMF_1D(t[1:],[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]))
        else:
            est_bt =  forward.laplace.ADEMF_1D(t,[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]
        if memory_func == 'power law':
            obj = -RMSE_func(t,btc,est_bt)-0.02*(v_eps-1)**2
            lipo_errors_list.append(obj)
            return obj
        else:
            obj = -RMSE_func(t,btc,est_bt) 
            lipo_errors_list.append(obj)
            return obj
    pre_eval_x = dict(v_eps = (best_params_no_syn[:,0]/vs_peak)[i_btc], Pe = best_params_no_syn[i_btc,1], beta_prime = best_params_no_syn[i_btc,2],mem_param = best_params_no_syn[i_btc,3])
    evaluations = [(pre_eval_x, opt_function(**pre_eval_x))]
    search = GlobalOptimizer(
        opt_function,
        lower_bounds={'v_eps': vs_eps_min,'Pe':np.min(best_params_no_syn[:,1]),'beta_prime':np.min(best_params_no_syn[:,2]), 'mem_param':np.min(best_params_no_syn[:,3])},
        upper_bounds={'v_eps': vs_eps_max,'Pe':np.max(best_params_no_syn[:,1]),'beta_prime':np.max(best_params_no_syn[:,2]), 'mem_param':np.max(best_params_no_syn[:,3])} ,
        evaluations=evaluations,
        maximize=True
    )
    search.run(num_function_calls)
    v_eps,Pe,beta_prime, mem_param = list(search.optimum[0].values())
    if t[0]==0:
        est_bt = np.concatenate(([0],forward.laplace.ADEMF_1D(t[1:],[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]))
    else:
        est_bt = forward.laplace.ADEMF_1D(t,[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]

    params_LIPO[i_btc] = np.array([vs_peak[i_btc]*v_eps,Pe,beta_prime, mem_param])
    RMSEs_LIPO[i_btc] = RMSE_func(t,btc,est_bt)
    KL_divs_LIPO[i_btc] = KL_div_func(t,btc,est_bt)
    btcs_LIPO.append(est_bt)
    lipo_errors_mat.append(lipo_errors_list)
    params_LIPO_steps.append(params_step)
    print(i_btc, params_LIPO[i_btc], RMSEs_LIPO[i_btc])

In [ ]:
if not only_params:
    lipo_errors_mat_corr = -np.array([[max(lipo_errors_mat[i_btc][:i+1]) for i in range(num_function_calls)] for  i_btc in range(n_btcs)])
    params_LIPO_steps_corr = -np.array([[params_LIPO_steps[i_btc][np.argmax(lipo_errors_mat[i_btc][:i+1])] for i in range(num_function_calls)] for i_btc in range(n_btcs)])
    plt.plot(np.exp(np.mean(np.log(lipo_errors_mat_corr),axis = 0)))

In [ ]:
def best_params_and_errors(list_params,list_errors):
    all_params = np.stack(list_params)
    all_errors = np.stack(list_errors)
    print(all_params.shape,all_errors.shape)
    print(np.argmin(all_errors, axis = 0))
    
    best_params = np.array([all_params[opt,i] for i,opt in enumerate(np.argmin(all_errors, axis = 0))])
    best_errors = np.array([all_errors[opt,i] for i,opt in enumerate(np.argmin(all_errors, axis = 0))])
    return best_params,best_errors

In [ ]:
if only_params:
    if memory_func == 'first order':
        best_params,best_RMSEs = best_params_and_errors([best_params_no_syn,params_PBI],[np.minimum(RMSEs_laplace,RMSEs_moments),RMSEs_PBI])
    else:
        best_params,best_RMSEs = best_params_and_errors([params_laplace,params_PBI],[RMSEs_laplace,RMSEs_PBI])
else:
    best_params,best_RMSEs = best_params_and_errors([params_LIPO,params_PBI],[RMSEs_LIPO,RMSEs_PBI])

In [ ]:
best_params = params_PBI.copy()

In [ ]:
%%time
params_LIPO_2 = np.zeros((n_btcs,4))
KL_divs_LIPO_2 = np.zeros((n_btcs))
RMSEs_LIPO_2 = np.zeros((n_btcs))
btcs_LIPO_2 = []
lipo_errors_mat_2 = []
params_LIPO_2_steps = []
num_function_calls = 300
for i_btc in range(n_btcs):
    lipo_errors_list = []
    params_step = []
    t,btc,x = ts_m_0[i_btc],btcs_m_0[i_btc],xs_m[i_btc]
    def opt_function(v_eps,Pe,beta_prime, mem_param):
        params_step.append([v_eps,Pe,beta_prime, mem_param])
        if t[0]==0:
            est_bt =  np.concatenate(([0],forward.laplace.ADEMF_1D(t[1:],[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]))
        else:
            est_bt =  forward.laplace.ADEMF_1D(t,[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]
        if memory_func == 'power law':
            obj = -RMSE_func(t,btc,est_bt)-0.02*(v_eps-1)**2
            lipo_errors_list.append(obj)
            return obj
        else:
            obj = -RMSE_func(t,btc,est_bt) 
            lipo_errors_list.append(obj)
            return obj
    pre_eval_x = dict(v_eps = (best_params[:,0]/vs_peak)[i_btc], Pe = best_params[i_btc,1], beta_prime = best_params[i_btc,2],mem_param = best_params[i_btc,3])
    evaluations = [(pre_eval_x, opt_function(**pre_eval_x))]
    search = GlobalOptimizer(
        opt_function,
        lower_bounds={'v_eps': (best_params[:,0]/vs_peak)[i_btc]*0.98,'Pe': best_params[i_btc,1]/1.3,'beta_prime': best_params[i_btc,2]/1.3, 'mem_param': best_params[i_btc,3]/1.3},
        upper_bounds={'v_eps': (best_params[:,0]/vs_peak)[i_btc]*1.02,'Pe': best_params[i_btc,1]*1.3,'beta_prime': best_params[i_btc,2]*1.3, 'mem_param': best_params[i_btc,3]*1.3} ,
        evaluations=evaluations,
        maximize=True
    )
    search.run(num_function_calls)
    v_eps,Pe,beta_prime, mem_param = list(search.optimum[0].values())
    if t[0]==0:
        est_bt = np.concatenate(([0],forward.laplace.ADEMF_1D(t[1:],[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]))
    else:
        est_bt = forward.laplace.ADEMF_1D(t,[x],vs_peak[i_btc]*v_eps, Pe,beta_prime,[1,mem_param],memory_func, bound_cond)[:,-1]

    params_LIPO_2[i_btc] = np.array([vs_peak[i_btc]*v_eps,Pe,beta_prime, mem_param])
    RMSEs_LIPO_2[i_btc] = RMSE_func(t,btc,est_bt)
    KL_divs_LIPO_2[i_btc] = KL_div_func(t,btc,est_bt)
    btcs_LIPO_2.append(est_bt)
    lipo_errors_mat_2.append(lipo_errors_list)
    params_LIPO_2_steps.append(params_step)
    print(i_btc, params_LIPO_2[i_btc], RMSEs_LIPO_2[i_btc])

In [ ]:
params_LIPO_2_steps_corr = -np.array([[params_LIPO_2_steps[i_btc][np.argmax(lipo_errors_mat_2[i_btc][:i+1])] for i in range(num_function_calls)] for i_btc in range(n_btcs)])
plt.plot(np.exp(np.mean(np.log(lipo_errors_mat_corr_2),axis = 0)))

# Save results

In [91]:
if only_params:
    results_dict = {'Estimated parameters':params_LIPO_2,
        'Errors in reconstruction of breakthrough curves':
                    {'RMSE':RMSEs_LIPO_2, 'KL divergence':KL_divs_LIPO_2}
    }
else:
    results_dict = {
        'Estimated parameters':{
            'Laplace':params_laplace,
            'NNI':params_NNI,
            'PBI':params_PBI,
            'Inverse KL-DNN':params_KL_DNN_inverse,
            'Inverse KL-DNN augmented':params_aug_KL_DNN_inverse,
            'KL-DNN-MAP':params_KL_DNN_MAP,
            'LIPO':params_LIPO,
            'LIPO, best initial':params_LIPO_2
        },
        'Errors in reconstruction of breakthrough curves':{
            'RMSE':{
                'Laplace':RMSEs_laplace,
                'NNI':RMSEs_NNI,
                'PBI':RMSEs_PBI,
                'Inverse KL-DNN':RMSEs_KL_DNN_inverse,
                'Inverse KL-DNN augmented':RMSEs_aug_KL_DNN_inverse,
                'KL-DNN-MAP':RMSEs_KL_DNN_MAP,
                'LIPO':RMSEs_LIPO,
                'LIPO, best initial':RMSEs_LIPO_2
            },
            'KL divergence':{
                'Laplace':KL_divs_laplace,
                'NNI':KL_divs_NNI,
                'PBI':KL_divs_PBI,
                'Inverse KL-DNN':KL_divs_KL_DNN_inverse,
                'Inverse KL-DNN augmented':KL_divs_aug_KL_DNN_inverse,
                'KL-DNN-MAP':KL_divs_KL_DNN_MAP,
                'LIPO': KL_divs_LIPO,
                'LIPO, best initial':KL_divs_LIPO_2
            }
        },
        'Train indices': indices_train
    }
    if memory_func == 'first order':
        results_dict['Estimated parameters']['Moments'] = params_moments
        results_dict['Errors in reconstruction of breakthrough curves']['RMSE']['Moments'] = RMSEs_moments
        results_dict['Errors in reconstruction of breakthrough curves']['KL divergence']['Moments'] = KL_divs_moments

In [92]:
with open('results-'+bound_cond+'-'+memory_func+'-'+seed+'_output.npy', 'wb') as f:
    np.save(f, results_dict)